In [1]:
import rasterio
import numpy as np
from pathlib import Path

dem_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/dem")  # adjust to your path

In [2]:
print(f"{'Patch':<25} {'CRS':>12} {'Res (deg)':>10} {'Min m':>8} {'Max m':>8} {'Mean m':>8} {'Nodata %':>9}")
print("-" * 85)

for f in sorted(dem_dir.glob("*.tif")):
    with rasterio.open(f) as src:
        data = src.read(1).astype(float)
        
        # Handle nodata
        if src.nodata is not None:
            valid = data[data != src.nodata]
        else:
            valid = data[data > -9999]
        
        print(f"{f.stem:<25} "
              f"{str(src.crs.to_epsg()):>12} "
              f"{src.res[0]:>10.5f} "
              f"{valid.min():>8.1f} "
              f"{valid.max():>8.1f} "
              f"{valid.mean():>8.1f} "
              f"{100*(1-len(valid)/data.size):>8.1f}%")

Patch                              CRS  Res (deg)    Min m    Max m   Mean m  Nodata %
-------------------------------------------------------------------------------------
P01_Kanto_dem                     4326    0.00028    -86.0   3756.0    314.5     21.6%
P02_Tohoku_dem                    4326    0.00028   -122.0   2015.0    213.8     38.9%
P03_Chile_dem                     4326    0.00028    -39.0   6100.0   1326.6      2.8%
P04_Turkey_dem                    4326    0.00028    -17.0   3072.0   1028.1      0.0%
P05_Nepal_dem                     4326    0.00028     55.0   7951.0   3150.5      0.0%
P06_NewZealand_dem                4326    0.00028    -40.0   2757.0    292.9      4.8%
P07_Sumatra_dem                   4326    0.00028    -78.0   3159.0    203.0      8.9%
P08_Kutch_dem                     4326    0.00028   -100.0   1063.0     44.6      2.4%
P09_Longmenshan_dem               4326    0.00028    243.0   6011.0   1788.9      0.0%
P10_Australia_dem                 4326    0.

In [3]:
patches_to_check = ["P01_Kanto_dem", "P02_Tohoku_dem", 
                    "P08_Kutch_dem", "P11_Norway_dem"]

for f in sorted(dem_dir.glob("*.tif")):
    if f.stem not in patches_to_check:
        continue
    with rasterio.open(f) as src:
        data = src.read(1).astype(float)
        if src.nodata is not None:
            data[data == src.nodata] = np.nan
        
        neg = data[data < 0]
        print(f"\n{f.stem}")
        print(f"  Negative pixels: {len(neg):,}")
        print(f"  Negative range:  {neg.min():.1f} – {neg.max():.1f} m")
        print(f"  % of total:      {100*len(neg)/data.size:.3f}%")
        
        # Where are they spatially
        neg_idx = np.where(data < 0)
        rows, cols = neg_idx
        transform = src.transform
        lons = transform.c + cols * transform.a
        lats = transform.f + rows * transform.e
        print(f"  Lon range of negatives: {lons.min():.2f} – {lons.max():.2f}")
        print(f"  Lat range of negatives: {lats.min():.2f} – {lats.max():.2f}")


P01_Kanto_dem
  Negative pixels: 288,633
  Negative range:  -86.0 – -1.0 m
  % of total:      0.275%
  Lon range of negatives: 138.50 – 140.99
  Lat range of negatives: 34.61 – 37.15

P02_Tohoku_dem
  Negative pixels: 54,026
  Negative range:  -122.0 – -1.0 m
  % of total:      0.046%
  Lon range of negatives: 140.79 – 142.07
  Lat range of negatives: 37.52 – 40.49

P08_Kutch_dem
  Negative pixels: 2,993,877
  Negative range:  -100.0 – -1.0 m
  % of total:      2.200%
  Lon range of negatives: 68.50 – 72.00
  Lat range of negatives: 21.50 – 24.50

P11_Norway_dem
  Negative pixels: 10,738
  Negative range:  -174.0 – -1.0 m
  % of total:      0.007%
  Lon range of negatives: 5.00 – 8.97
  Lat range of negatives: 58.50 – 60.97


## Insights

<p>The DEM data was sourced from OpenTopography's NASADEM product, delivered as individual per-patch GeoTIFFs at approximately 0.00028° resolution (~30m) in WGS84/EPSG:4326. It is consistent with all other layers and requiring no reprojection. This is by far the highest resolution static layer in the dataset, approximately 360× finer than CRUST1.0, and will provide the sharpest spatial detail in the frozen prior after resampling to the 0.1° processing grid.</p>

<p>Elevation means and ranges across patches are physically precise and ordered as expected. Nepal records the highest mean elevation at 3,150m with a maximum of 7,951m: the near-Everest terrain of the central Himalayas, with the true summit just outside the patch boundary. Longmenshan follows at 1,789m mean with a 6,011m maximum, reflecting the dramatic transition from the Sichuan Basin floor (~500m) to the Tibetan Plateau margin (~4,000–6,000m) within a single patch which is the largest elevation gradient of any patch and a geologically meaningful signal for the clustering step. Ordos records a consistent 1,274m mean with a narrow 549–1,906m range, confirming the flat Loess Plateau character already suggested by the sediment and crustal thickness layers. Western Australia is the flattest patch by a significant margin, a 167–695m range and 384m mean with zero nodata, classic stable craton topography with no tectonic relief.</p>

<p>Nodata percentages vary substantially and reflect ocean coverage rather than data gaps. Tohoku (38.9%) and Kanto (21.6%) show the highest nodata fractions due to their significant Pacific Ocean coverage, which the land-surface DEM correctly masks. Norway (16.7%) reflects extensive fjord and coastal water coverage. These masked cells will be handled consistently during the rasterisation step and do not affect land-surface feature quality.</p>

<p>Negative elevation values were identified in four patches and require patch-specific treatment during processing. Kanto (0.275%) and Tohoku (0.046%) show small negative values confined to the immediate coastline, shallow nearshore bathymetric pixels bleeding through the land mask, which will be clipped to zero during processing. Norway (0.007%, minimum -174m) captures genuine fjord floors that are physically real but below sea level. Also its clipped to zero as the model is concerned with land surface conditions. Kutch is the exception: 2.2% of pixels are negative, spanning the full patch extent rather than being confined to the coast. This reflects the Rann of Kutch: a large seasonal salt marsh and sedimentary basin that sits genuinely below sea level across a wide area. These negative values are retained as-is during processing as they represent a physically meaningful terrain feature that distinguishes the Kutch patch from all others and contributes useful signal to the geological clustering.</p>

<p>Across all four static layers inspected so far: Vs30, sediment thickness, crustal thickness, and DEM; a consistent multi-layer picture of geological regime is emerging. The collision and plateau patches (Nepal, Sichuan, Ordos) show deep Moho, thick sediment, high elevation, and moderate-low Vs30. The subduction patches (Kanto, Tohoku, Sumatra) show thin crust, high within-patch elevation variance, and significant ocean fractions. The stable craton patches (Australia, Ordos, Norway) show consistently narrow feature ranges across all layers, reflecting tectonic quiescence expressed simultaneously in crustal structure, sediment cover, surface topography, and site conditions. This cross-layer coherence is an encouraging early signal that the GMM clustering will recover physically interpretable regime boundaries from the combined static feature space.</p>